In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("bronze_ais_events")

In [0]:
print(f"Bronze rows: {bronze_df.count()}")
bronze_df.printSchema()
display(bronze_df.limit(10))

In [0]:
silver_df = (
    bronze_df.filter(
        (F.col("latitude").between(-90, 90)) &
        (F.col("longitude").between(-180, 180))
    )
)
print(f"Silver rows after geo validation: {silver_df.count()}")

In [0]:
silver_df = silver_df.withColumn(
    "event_date",
    F.to_date("event_timestamp")
)
display(
    silver_df.select("event_timestamp", "event_date").limit(10)
)

In [0]:
from pyspark.sql import functions as F

duplicate_count = (
    bronze_df
    .groupBy("vessel_id","event_timestamp","latitude","longitude","speed_knots","heading")
    .count()
    .filter(F.col("count")>1)
    .count()
)

print(f"Duplicate groups: {duplicate_count}")

In [0]:
silver_df = (
    bronze_df
    .filter(
        (F.col("latitude").between(-90, 90)) &
        (F.col("longitude").between(-180, 180))
    )
    .dropDuplicates(["vessel_id", "event_timestamp", "latitude", "longitude"])
    .withColumn("event_date", F.to_date("event_timestamp"))
    .withColumn("event_hour", F.hour("event_timestamp"))
    .withColumn(
        "speed_category",
        F.when(F.col("speed_knots") == 0, "stationary")
         .when(F.col("speed_knots") < 5, "slow")
         .when(F.col("speed_knots") < 15, "medium")
         .otherwise("fast")
    )
    .withColumn(
        "is_moving",
        F.col("speed_knots") > 0
    )
)

In [0]:
print(f"Silver rows: {silver_df.count()}")
display(
    silver_df.select(
        "vessel_id",
        "event_timestamp",
        "event_date",
        "event_hour",
        "speed_knots",
        "speed_category",
        "is_moving"
    ).limit(10)
)

In [0]:
silver_df.write\
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_ais_events")

In [0]:
spark.table("silver_ais_events").printSchema()
display(spark.table("silver_ais_events").limit(10))